In [ ]:

import pandas as pd
import time
import json
import re
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def analyze_movie_title_realistic(title):
    prompt = f"""
    Analyze this movie title: "{title}"

    Provide a JSON response with these keys:
    - detected_language
    - transliteration
    - translation
    - confidence (must be: high, medium, or low)
    - language_family

    Be realistic about confidence levels based on title clarity.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a language analyst. Assign realistic confidence levels. Short or ambiguous titles should get lower confidence."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )

        result_text = response.choices[0].message.content
        json_match = re.search(r'\{[^}]+\}', result_text)
        if json_match:
            return json.loads(json_match.group())
        return {"error": "Parse error"}
    except Exception as e:
        return {"error": str(e)}

# Titles that naturally create confidence variation
titles = [
    # Clear, long titles
    "The Shawshank Redemption", "Pulp Fiction: The Movie", "The Lord of the Rings: Return of the King",

    # Foreign language titles
    "Le Fabuleux Destin d'Amélie Poulain", "El laberinto del fauno", "La vita è bella",

    # Medium difficulty
    "Amélie", "Parasite", "Drive", "Gravity", "Moonlight",

    # Short/ambiguous titles
    "Heat", "Crash", "Her", "Moon", "Up", "It", "Roma", "Paris"
]

print("Processing titles with realistic confidence...")
results = []
for i, title in enumerate(titles):
    print(f"{i+1}/{len(titles)}: {title}")
    analysis = analyze_movie_title_realistic(title)
    analysis['original_title'] = title
    analysis['word_count'] = len(title.split())
    results.append(analysis)

    # Show confidence immediately
    conf = analysis.get('confidence', 'unknown')
    print(f"   Confidence: {conf}")
    time.sleep(0.5)

df = pd.DataFrame(results)
df.to_csv('realistic_analysis.csv', index=False)

print(f"\nFinal confidence distribution:")
print(df['confidence'].value_counts())

Processing titles with realistic confidence...
1/19: The Shawshank Redemption
   Confidence: high
2/19: Pulp Fiction: The Movie
   Confidence: high
3/19: The Lord of the Rings: Return of the King
   Confidence: high
4/19: Le Fabuleux Destin d'Amélie Poulain
   Confidence: high
5/19: El laberinto del fauno
   Confidence: high
6/19: La vita è bella
   Confidence: high
7/19: Amélie
   Confidence: high
8/19: Parasite
   Confidence: high
9/19: Drive
   Confidence: medium
10/19: Gravity
   Confidence: high
11/19: Moonlight
   Confidence: high
12/19: Heat
   Confidence: medium
13/19: Crash
   Confidence: medium
14/19: Her
   Confidence: high
15/19: Moon
   Confidence: medium
16/19: Up
   Confidence: medium
17/19: It
   Confidence: low
18/19: Roma
   Confidence: medium
19/19: Paris
   Confidence: medium

Final confidence distribution:
confidence
high      11
medium     7
low        1
Name: count, dtype: int64


In [ ]:

import pandas as pd
import time
import json
import re
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def analyze_movie_title_realistic(title):
    prompt = f"""
    Analyze this movie title: "{title}"

    Provide a JSON response with these keys:
    - detected_language
    - transliteration
    - translation
    - confidence (must be: high, medium, or low)
    - language_family

    Be realistic about confidence levels based on title clarity.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a language analyst. Assign realistic confidence levels. Short or ambiguous titles should get lower confidence."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )

        result_text = response.choices[0].message.content
        json_match = re.search(r'\{[^}]+\}', result_text)
        if json_match:
            return json.loads(json_match.group())
        return {"error": "Parse error"}
    except Exception as e:
        return {"error": str(e)}

# 150+ NEW MOVIE TITLES FOR RICHARD'S ANALYSIS
titles = [
    # === CLEAR/LONG TITLES (Should get HIGH confidence) ===
    "The Pursuit of Happyness", "There Will Be Blood", "No Country for Old Men",
    "The Curious Case of Benjamin Button", "The Assassination of Jesse James",
    "Eternal Sunshine of the Spotless Mind", "The Grand Budapest Hotel",
    "The Shape of Water", "Three Billboards Outside Ebbing Missouri",
    "The Fabelmans", "Everything Everywhere All at Once", "The Banshees of Inisherin",
    "All Quiet on the Western Front", "The Power of the Dog", "Nomadland",
    "The Father", "Promising Young Woman", "The Trial of the Chicago 7",
    "Mank", "Minari", "Sound of Metal", "The Irishman", "Once Upon a Time in Hollywood",
    "Parasite", "Jojo Rabbit", "Little Women", "Marriage Story", "1917",
    "The Two Popes", "Joker", "The Lighthouse", "Portrait of a Lady on Fire",

    # === FOREIGN LANGUAGE TITLES (Mixed confidence) ===
    "Roma", "Cold War", "Shoplifters", "Burning", "Capernaum", "The Great Beauty",
    "The Hunt", "The Broken Circle Breakdown", "The Missing Picture", "The Square",
    "A Fantastic Woman", "On Body and Soul", "The Salesman", "The Teacher",
    "Toni Erdmann", "The Club", "Son of Saul", "Mustang", "Leviathan",
    "Ida", "The Great Beauty", "The Broken Circle Breakdown", "The Missing Picture",

    # === MEDIUM DIFFICULTY TITLES ===
    "Drive", "Nightcrawler", "Prisoners", "Sicario", "Arrival", "Ex Machina",
    "Her", "Moon", "Sunshine", "Limitless", "Source Code", "Looper", "Predestination",
    "Coherence", "The Invitation", "It Follows", "The Babadook", "The Witch",
    "A Quiet Place", "Get Out", "Us", "Hereditary", "Midsommar", "The Lighthouse",

    # === SHORT/AMBIGUOUS TITLES (Should get LOW confidence) ===
    "Heat", "Crash", "Traffic", "Juno", "Ray", "Ali", "Milk", "Lincoln", "Capote",
    "Zodiac", "Se7en", "Memento", "Gravity", "Her", "Moon", "Up", "It", "The",
    "A", "I", "You", "We", "Us", "Cars", "Jaws", "Frozen", "Brave", "Tangled",
    "Shrek", "Zootopia", "Coco", "Moana", "Soul", "Raya", "Encanto", "Frozen",
    "Cars", "Toys", "Cars", "Coco", "Soul", "Up", "Cars", "Ratatouille", "Bolt",
    "Rio", "Nemo", "Coco", "Frozen", "Brave", "Tangled", "Shrek", "Zootopia"
]

print("Processing 150+ titles with realistic confidence...")
results = []
for i, title in enumerate(titles):
    print(f"Processing {i+1}/{len(titles)}: {title}")
    analysis = analyze_movie_title_realistic(title)
    analysis['original_title'] = title
    analysis['word_count'] = len(title.split())
    results.append(analysis)

    # Show confidence immediately
    conf = analysis.get('confidence', 'unknown')
    print(f"   Confidence: {conf}")
    time.sleep(0.5)

df = pd.DataFrame(results)
df.to_csv('richard_analysis_150_titles.csv', index=False)

print(f"\nFinal confidence distribution:")
print(df['confidence'].value_counts())


Processing 150+ titles with realistic confidence...
Processing 1/132: The Pursuit of Happyness
   Confidence: high
Processing 2/132: There Will Be Blood
   Confidence: high
Processing 3/132: No Country for Old Men
   Confidence: high
Processing 4/132: The Curious Case of Benjamin Button
   Confidence: high
Processing 5/132: The Assassination of Jesse James
   Confidence: high
Processing 6/132: Eternal Sunshine of the Spotless Mind
   Confidence: high
Processing 7/132: The Grand Budapest Hotel
   Confidence: high
Processing 8/132: The Shape of Water
   Confidence: high
Processing 9/132: Three Billboards Outside Ebbing Missouri
   Confidence: high
Processing 10/132: The Fabelmans
   Confidence: high
Processing 11/132: Everything Everywhere All at Once
   Confidence: high
Processing 12/132: The Banshees of Inisherin
   Confidence: high
Processing 13/132: All Quiet on the Western Front
   Confidence: high
Processing 14/132: The Power of the Dog
   Confidence: high
Processing 15/132: Nomadl

In [ ]:

df = pd.read_csv('richard_analysis_150_titles.csv')

print("LANGUAGE DETECTION ANALYSIS RESULTS")
print("=" * 50)

# 1. Success/Failure Analysis
total_titles = len(df)
successful_analyses = len(df[df['detected_language'].notna()])
failed_analyses = len(df[df['detected_language'].isna()])

print(f"1. SUCCESS RATE ANALYSIS:")
print(f"   Total titles processed: {total_titles}")
print(f"   Successful analyses: {successful_analyses} ({(successful_analyses/total_titles)*100:.1f}%)")
print(f"   Failed analyses: {failed_analyses} ({(failed_analyses/total_titles)*100:.1f}%)")

# Show what failed
if failed_analyses > 0:
    failed_titles = df[df['detected_language'].isna()]
    print(f"   Failed titles sample:")
    for title in failed_titles['original_title'].head(5):
        print(f"     - {title}")

# 2. Confidence Distribution (only for successful analyses)
successful_df = df[df['detected_language'].notna()]

if len(successful_df) > 0:
    conf_dist = successful_df['confidence'].value_counts()
    print(f"\n2. CONFIDENCE LEVEL DISTRIBUTION:")
    for level, count in conf_dist.items():
        percentage = (count / len(successful_df)) * 100
        print(f"   {level}: {count} titles ({percentage:.1f}%)")

    # 3. Title Length vs Confidence (Richard's "tipping point")
    print(f"\n3. TITLE LENGTH IMPACT ANALYSIS:")
    short_titles = successful_df[successful_df['word_count'] <= 2]
    long_titles = successful_df[successful_df['word_count'] > 2]

    if len(short_titles) > 0:
        short_high = len(short_titles[short_titles['confidence'] == 'high']) / len(short_titles) * 100
        print(f"   Short titles (1-2 words): {len(short_titles)} samples")
        print(f"     High confidence: {short_high:.1f}%")

    if len(long_titles) > 0:
        long_high = len(long_titles[long_titles['confidence'] == 'high']) / len(long_titles) * 100
        print(f"   Long titles (3+ words): {len(long_titles)} samples")
        print(f"     High confidence: {long_high:.1f}%")

    if len(short_titles) > 0 and len(long_titles) > 0:
        improvement = long_high - short_high
        print(f"   PERFORMANCE DIFFERENCE: +{improvement:.1f}% for longer titles")

    # 4. Language Family Performance
    print(f"\n4. LANGUAGE FAMILY PERFORMANCE:")
    if 'language_family' in successful_df.columns:
        families = successful_df['language_family'].value_counts().head(5)
        for family, count in families.items():
            family_data = successful_df[successful_df['language_family'] == family]
            high_conf_rate = len(family_data[family_data['confidence'] == 'high']) / len(family_data) * 100
            print(f"   {family}: {count} titles, {high_conf_rate:.1f}% high confidence")

    # 5. Where AI Struggles (Low Confidence Analysis)
    low_conf_titles = successful_df[successful_df['confidence'] == 'low']
    if len(low_conf_titles) > 0:
        print(f"\n5. CHALLENGING TITLES (Low Confidence):")
        print(f"   Found {len(low_conf_titles)} titles with low confidence")
        print(f"   Average word count: {low_conf_titles['word_count'].mean():.1f}")
        print(f"   Sample titles:")
        for title in low_conf_titles['original_title'].head(8):
            print(f"     - {title}")

LANGUAGE DETECTION ANALYSIS RESULTS
1. SUCCESS RATE ANALYSIS:
   Total titles processed: 132
   Successful analyses: 132 (100.0%)
   Failed analyses: 0 (0.0%)

2. CONFIDENCE LEVEL DISTRIBUTION:
   high: 89 titles (67.4%)
   medium: 36 titles (27.3%)
   low: 7 titles (5.3%)

3. TITLE LENGTH IMPACT ANALYSIS:
   Short titles (1-2 words): 103 samples
     High confidence: 58.3%
   Long titles (3+ words): 29 samples
     High confidence: 100.0%
   PERFORMANCE DIFFERENCE: +41.7% for longer titles

4. LANGUAGE FAMILY PERFORMANCE:
   Germanic: 110 titles, 67.3% high confidence
   Romance: 12 titles, 66.7% high confidence
   Koreanic: 2 titles, 100.0% high confidence
   Uralic: 2 titles, 100.0% high confidence
   Austronesian: 2 titles, 50.0% high confidence

5. CHALLENGING TITLES (Low Confidence):
   Found 7 titles with low confidence
   Average word count: 1.1
   Sample titles:
     - The Club
     - It
     - The
     - A
     - I
     - You
     - We


In [ ]:
# === ANALYSIS FOR 10 CHALLENGING TITLES ===
import pandas as pd
import time
import json
import re
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def analyze_movie_title_realistic(title):
    prompt = f"Analyze this movie title: '{title}'. Provide JSON with: detected_language, transliteration, translation, confidence, language_family."

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )

        result_text = response.choices[0].message.content
        json_match = re.search(r'\{[^}]+\}', result_text)
        if json_match:
            return json.loads(json_match.group())
        return {"error": "Parse error"}
    except Exception as e:
        return {"error": str(e)}

# 10 CHALLENGING TITLES
challenging_titles = [
    "Ø", "Ψ", "Æ", "Œ", "ꙮ", "𖥸", "𓂀", "🜄", "ᚻ", "꧁"
]

print("Processing 10 challenging titles...")
results = []
for i, title in enumerate(challenging_titles):
    print(f"Processing {i+1}/10: '{title}'")
    analysis = analyze_movie_title_realistic(title)
    analysis['original_title'] = title
    analysis['word_count'] = len(title.split())
    results.append(analysis)

    conf = analysis.get('confidence', 'unknown')
    print(f"   Confidence: {conf}")
    time.sleep(0.5)

df = pd.DataFrame(results)
df.to_csv('challenging_titles_analysis.csv', index=False)

# === RICHARD'S ANALYSIS FORMAT ===
print("\n" + "="*50)
print("LANGUAGE DETECTION ANALYSIS RESULTS")
print("="*50)

# 1. Success/Failure Analysis
total_titles = len(df)
successful_analyses = len(df[df['detected_language'].notna()])
failed_analyses = len(df[df['detected_language'].isna()])

print(f"1. SUCCESS RATE ANALYSIS:")
print(f"   Total titles processed: {total_titles}")
print(f"   Successful analyses: {successful_analyses} ({(successful_analyses/total_titles)*100:.1f}%)")
print(f"   Failed analyses: {failed_analyses} ({(failed_analyses/total_titles)*100:.1f}%)")

# Show what failed
if failed_analyses > 0:
    failed_titles = df[df['detected_language'].isna()]
    print(f"   Failed titles sample:")
    for title in failed_titles['original_title'].head(8):
        print(f"     - {title}")

# 2. Confidence Distribution
successful_df = df[df['detected_language'].notna()]

if len(successful_df) > 0:
    conf_dist = successful_df['confidence'].value_counts()
    print(f"\n2. CONFIDENCE LEVEL DISTRIBUTION:")
    for level, count in conf_dist.items():
        percentage = (count / len(successful_df)) * 100
        print(f"   {level}: {count} titles ({percentage:.1f}%)")

# 3. Title Length vs Confidence
print(f"\n3. TITLE LENGTH IMPACT ANALYSIS:")
short_titles = successful_df[successful_df['word_count'] <= 2]
long_titles = successful_df[successful_df['word_count'] > 2]

if len(short_titles) > 0:
    short_high = len(short_titles[short_titles['confidence'] == 'high']) / len(short_titles) * 100
    print(f"   Short titles (1-2 words): {len(short_titles)} samples")
    print(f"     High confidence: {short_high:.1f}%")

if len(long_titles) > 0:
    long_high = len(long_titles[long_titles['confidence'] == 'high']) / len(long_titles) * 100
    print(f"   Long titles (3+ words): {len(long_titles)} samples")
    print(f"     High confidence: {long_high:.1f}%")

if len(short_titles) > 0 and len(long_titles) > 0:
    improvement = long_high - short_high
    print(f"   PERFORMANCE DIFFERENCE: +{improvement:.1f}% for longer titles")

# 4. Language Family Performance
print(f"\n4. LANGUAGE FAMILY PERFORMANCE:")
if 'language_family' in successful_df.columns:
    families = successful_df['language_family'].value_counts().head(5)
    for family, count in families.items():
        family_data = successful_df[successful_df['language_family'] == family]
        high_conf_rate = len(family_data[family_data['confidence'] == 'high']) / len(family_data) * 100
        print(f"   {family}: {count} titles, {high_conf_rate:.1f}% high confidence")

# 5. Challenging Titles Analysis
low_conf_titles = successful_df[successful_df['confidence'] == 'low']
if len(low_conf_titles) > 0:
    print(f"\n5. CHALLENGING TITLES (Low Confidence):")
    print(f"   Found {len(low_conf_titles)} titles with low confidence")
    print(f"   Average word count: {low_conf_titles['word_count'].mean():.1f}")
    print(f"   Sample titles:")
    for title in low_conf_titles['original_title'].head(8):
        print(f"     - {title}")

Processing 10 challenging titles...
Processing 1/10: 'Ø'
   Confidence: 0.85
Processing 2/10: 'Ψ'
   Confidence: 0.95
Processing 3/10: 'Æ'
   Confidence: 0.85
Processing 4/10: 'Œ'
   Confidence: 0.85
Processing 5/10: 'ꙮ'
   Confidence: 0.95
Processing 6/10: '𖥸'
   Confidence: 0.85
Processing 7/10: '𓂀'
   Confidence: 0.95
Processing 8/10: '🜄'
   Confidence: 0.95
Processing 9/10: 'ᚻ'
   Confidence: 0.95
Processing 10/10: '꧁'
   Confidence: 0.0

LANGUAGE DETECTION ANALYSIS RESULTS
1. SUCCESS RATE ANALYSIS:
   Total titles processed: 10
   Successful analyses: 10 (100.0%)
   Failed analyses: 0 (0.0%)

2. CONFIDENCE LEVEL DISTRIBUTION:
   0.95: 5 titles (50.0%)
   0.85: 4 titles (40.0%)
   0.0: 1 titles (10.0%)

3. TITLE LENGTH IMPACT ANALYSIS:
   Short titles (1-2 words): 10 samples
     High confidence: 0.0%

4. LANGUAGE FAMILY PERFORMANCE:
   Germanic: 3 titles, 0.0% high confidence
   Indo-European: 1 titles, 0.0% high confidence
   Romance: 1 titles, 0.0% high confidence
   Slavic: 1 t